# Task 2 · Unemployment Analysis with Python
**Objective:** Exploratory data analysis of unemployment data to uncover regional and temporal trends, focusing on the impact of COVID-19 on unemployment in India.

**Dataset:** *Unemployment in India* (Kaggle). Download `Unemployment in India.csv` and place it next to this notebook.

## 1. Imports and setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Data loading, shape inspection, null check and type conversion

In [ ]:
# Change the filename if yours differs
df = pd.read_csv("Unemployment in India.csv")

# Column names in this dataset often have stray spaces; clean them
df.columns = df.columns.str.strip()
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nNull values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Drop fully empty rows (the raw file sometimes has blank trailing rows)
df = df.dropna(how="all").copy()

# Rename to short, easy-to-type names
df = df.rename(columns={
    "Region": "State",
    "Estimated Unemployment Rate (%)": "Unemployment",
    "Estimated Employed": "Employed",
    "Estimated Labour Participation Rate (%)": "LabourParticipation",
})

# Type conversion: strip whitespace in text columns, parse dates (dd-mm-yyyy)
df["State"] = df["State"].str.strip()
if "Frequency" in df.columns:
    df["Frequency"] = df["Frequency"].str.strip()
df["Date"] = pd.to_datetime(df["Date"].str.strip(), dayfirst=True)

# Handle any remaining nulls in numeric columns
num_cols = ["Unemployment", "Employed", "LabourParticipation"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=num_cols)

# Helper time columns
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.strftime("%b")
df["Year"] = df["Date"].dt.year

print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("States/regions:", df["State"].nunique())
df.describe()

**Observations:** The dataset has monthly records per state with the unemployment rate, number employed and labour participation rate. After cleaning column names and converting `Date` to datetime, the data is ready for analysis.

## 3. EDA: region-wise averages and month-wise trends

In [ ]:
state_avg = (df.groupby("State")[num_cols]
               .mean()
               .sort_values("Unemployment", ascending=False)
               .round(2))
state_avg

In [ ]:
# Month-wise (chronological) national average
monthly = df.groupby("Date")["Unemployment"].mean()

plt.figure()
monthly.plot(marker="o", color="crimson")
plt.title("Average unemployment rate across all states, month by month")
plt.xlabel("Month")
plt.ylabel("Unemployment rate (%)")
plt.show()

**Observations:** The national average stays fairly steady through 2019 and then spikes sharply around April to May 2020, which matches the COVID-19 lockdown period. *(Edit these notes after checking your own output.)*

## 4. Time-series line chart for major states

In [ ]:
states = ["Maharashtra", "Delhi", "Tamil Nadu", "Uttar Pradesh", "Bihar"]  # change as you like
subset = df[df["State"].isin(states)]

plt.figure()
sns.lineplot(data=subset, x="Date", y="Unemployment", hue="State", marker="o")
plt.axvline(pd.Timestamp("2020-03-25"), color="black", linestyle="--", label="National lockdown (25 Mar 2020)")
plt.title("Unemployment rate over time: selected states")
plt.ylabel("Unemployment rate (%)")
plt.legend()
plt.show()

**Observations:** All the selected states jump after the lockdown began, but the size of the rise differs. Some states, such as Delhi, rise much more than others. *(Refine using your chart.)*

## 5. Bar chart: top 10 states by average unemployment rate

In [ ]:
top10 = state_avg["Unemployment"].head(10)

plt.figure()
sns.barplot(x=top10.values, y=top10.index, palette="Reds_r")
plt.title("Top 10 states by average unemployment rate")
plt.xlabel("Average unemployment rate (%)")
plt.ylabel("State")
for i, v in enumerate(top10.values):
    plt.text(v + 0.2, i, f"{v:.1f}", va="center")
plt.show()

**Observations:** The top 10 list shows which states had the highest average unemployment over the whole period. *(Name the top 3 states and their rates from your output.)*

## 6. Heatmap: unemployment, employment and labour participation

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation: unemployment, employment, labour participation")
plt.show()

**Observations:** Unemployment and labour participation tend to move in opposite directions in this data, since lockdown-era job losses coincide with people leaving the labour force. *(Confirm the signs and strengths from your heatmap.)*

## 7. Pre-COVID vs post-COVID comparison

In [ ]:
# Split at the start of India's national lockdown
cutoff = pd.Timestamp("2020-03-25")
df["Period"] = np.where(df["Date"] < cutoff, "Pre-COVID", "Post-COVID")

period_mean = df.groupby("Period")[num_cols].mean().round(2)
print(period_mean)

plt.figure(figsize=(7, 5))
sns.barplot(data=df, x="Period", y="Unemployment", order=["Pre-COVID", "Post-COVID"],
            estimator=np.mean, errorbar=None, palette=["steelblue", "tomato"])
plt.title("Mean unemployment rate: pre-COVID vs post-COVID")
plt.ylabel("Unemployment rate (%)")
plt.show()

In [ ]:
# State-wise change
pivot = df.pivot_table(index="State", columns="Period", values="Unemployment", aggfunc="mean")
pivot["Change"] = pivot["Post-COVID"] - pivot["Pre-COVID"]
pivot.sort_values("Change", ascending=False).round(2).head(10)

**Observations:** The mean unemployment rate roughly doubles or more after the lockdown began. The table above lists the states hit hardest. *(Quote your own numbers.)*

**Caveat:** The post-COVID window in this dataset is only a few months long, so it captures the lockdown shock, not the longer-term recovery.

## 8. Conclusions
1. Unemployment was relatively stable before March 2020, then rose sharply during the lockdown (peak around April to May 2020).
2. States differ widely in both their average unemployment and how hard COVID hit them.
3. Unemployment moves opposite to labour participation here, which suggests many people left the workforce instead of just becoming jobless.
4. The pre vs post comparison shows a large jump in the mean rate, though the post-COVID sample is short.